# Bölüm 15: Uygulama Geliştirme

> "Bilgi, pratiğe dökmediğiniz sürece değersizdir."
> — **Anton Çehov**, Yazar

---

## Öğrenecekleriniz

- Konuşma geçmişine sahip basit bir sohbet uygulaması nasıl geliştirilir
- Modellere dış bilgiye erişim sağlayan RAG (Erişimle Zenginleştirilmiş Üretim) deseni
- Belgeleri parçalama, gömme oluşturma ve ilgili bilgileri erişim yöntemleri
- Modellerin eylem almasını sağlayan temel araç çağırma desenleri
- Uygulamanızın gerçekten çalışıp çalışmadığını test etmek için değerlendirme stratejileri

---

## Kurulum

İlk olarak, gerekli paketleri yükleyelim ve ücretsiz yerel LLM çıkarımı için **Ollama**'yı kuralım.

> **Neden Ollama?** Tamamen ücretsiz, çevrimdışı çalışır ve herhangi bir bilgisayarda çalışır.
> API anahtarı veya kredi kartı gerektirmez. Birçok üretim uygulaması artık gizlilik ve
> maliyet tasarrufu için yerel modeller kullanıyor.

In [ ]:
# Gerekli paketleri yükle
!pip install -q sentence-transformers numpy requests

# === OLLAMA KURULUMU ===
# Ollama yerel olarak çalışır - tamamen ücretsiz, API anahtarı gerekmez!

print("Ollama yükleniyor...")
!curl -fsSL https://ollama.com/install.sh | sh

# Ollama sunucusunu arka planda başlat
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import time
time.sleep(3)  # Sunucunun başlaması için bekle

# Küçük bir model indir (~2GB indirme, tek seferlik)
print("\nllama3.2 modeli çekiliyor (ilk çalıştırmada birkaç dakika sürebilir)...")
!ollama pull llama3.2

# Yardımcı kütüphanemizi indir
!wget -q https://raw.githubusercontent.com/FirstLLM/code/main/llm_helper.py

print("\n✓ Kurulum tamamlandı! Artık ücretsiz yerel LLM'leri kullanabilirsiniz.")

In [ ]:
# ===== İÇE AKTARMALAR =====
import os
import json
import re
from datetime import datetime

import numpy as np

# LLM yardımcımızı içe aktar
from llm_helper import chat, chat_with_history
print("✓ llm_helper yüklendi")

# sentence-transformers'ın kullanılabilir olup olmadığını kontrol et (RAG gömmeleri için)
try:
    from sentence_transformers import SentenceTransformer
    print("✓ sentence-transformers yüklendi")
except ImportError:
    print("⚠ sentence-transformers bulunamadı. Şunu çalıştırın: pip install sentence-transformers")

In [ ]:
# ===== OLLAMA BAĞLANTISINI TEST ET =====
# Ollama'nın çalıştığını ve modelin kullanılabilir olduğunu doğrula

print("Ollama bağlantısı test ediliyor...")
try:
    response = chat("Sadece 'Bağlantı başarılı!' de ve başka bir şey söyleme.", temperature=0)
    print(f"✓ Ollama çalışıyor!")
    print(f"  Yanıt: {response}")
except Exception as e:
    print(f"⚠ Ollama bağlantısı başarısız: {e}")
    print("  Ollama'nın çalıştığından emin olun: ollama serve")

## 1. Sohbet Döngüsü Geliştirme

Konuşma geçmişini hatırlayan basit bir sohbet uygulamasıyla başlayalım.

In [ ]:
class ChatSession:
    """Geçmiş yönetimine sahip sohbet oturumu."""
    
    def __init__(self, max_history=20):
        self.max_history = max_history
        self.history = []
        self.system_prompt = "Sen yardımcı bir asistansın."
    
    def count_messages(self):
        """Konuşma geçmişindeki mesajları say."""
        return len(self.history)
    
    def trim_history_if_needed(self):
        """Limiti aşıyorsak en eski mesajları kaldır."""
        while len(self.history) > self.max_history:
            # En eski kullanıcı/asistan çiftini kaldır
            self.history.pop(0)
            if self.history and self.history[0]["role"] == "assistant":
                self.history.pop(0)
    
    def chat(self, user_message):
        """Bir kullanıcı mesajını işle ve yanıtı döndür."""
        # Kullanıcı mesajını ekle
        self.history.append({"role": "user", "content": user_message})
        
        # Gerekirse kırp
        self.trim_history_if_needed()
        
        # Yardımcımızı kullanarak yanıt al
        assistant_message = chat_with_history(
            self.history,
            system=self.system_prompt
        )
        
        # Geçmişe ekle
        self.history.append({"role": "assistant", "content": assistant_message})
        
        return assistant_message

print("ChatSession sınıfı tanımlandı!")

In [ ]:
# Sohbet oturumunu test et
session = ChatSession()

# İlk mesaj
response1 = session.chat("Merhaba! 2+2 kaçtır?")
print(f"Siz: Merhaba! 2+2 kaçtır?")
print(f"Asistan: {response1}")
print(f"Geçmiş mesajları: {session.count_messages()}\n")

# Takip sorusu (hafızayı gösterir)
response2 = session.chat("Az önce sana ne sordum?")
print(f"Siz: Az önce sana ne sordum?")
print(f"Asistan: {response2}")
print(f"Geçmiş mesajları: {session.count_messages()}")

## 2. RAG: Erişimle Zenginleştirilmiş Üretim

RAG, modelinize eğitilmediği belgelere erişim sağlar.

Bunu **açık kitap sınavı** gibi düşünün: model sadece hafızasına güvenmek yerine bilgileri arayabilir.

In [ ]:
class SimpleRAG:
    """Belge erişimi için basit bir RAG sistemi."""
    
    def __init__(self, embedding_model="all-MiniLM-L6-v2"):
        """RAG sistemini başlat."""
        self.encoder = SentenceTransformer(embedding_model)
        self.documents = []  # Orijinal metin parçaları
        self.embeddings = None  # Vektörlerin NumPy dizisi
        self.metadata = []  # Alıntılar için kaynak bilgisi
    
    def chunk_text(self, text, chunk_size=200, overlap=50):
        """Metni örtüşen parçalara böl."""
        words = text.split()
        chunks = []
        
        for i in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        
        return chunks
    
    def add_document(self, text, source_name="bilinmeyen"):
        """Bilgi tabanına bir belge ekle."""
        chunks = self.chunk_text(text)
        
        for i, chunk in enumerate(chunks):
            self.documents.append(chunk)
            self.metadata.append({
                "source": source_name,
                "chunk_index": i
            })
        
        # Tüm belgeleri yeniden göm
        self.embeddings = self.encoder.encode(
            self.documents,
            normalize_embeddings=True  # Kosinüs benzerliği için önemli
        )
        
        print(f"'{source_name}' kaynağından {len(chunks)} parça eklendi")
    
    def retrieve(self, query, top_k=3, min_score=0.3):
        """Bir sorgu için en ilgili parçaları bul."""
        if self.embeddings is None or len(self.embeddings) == 0:
            return []
        
        # Sorguyu göm
        query_embedding = self.encoder.encode(
            query,
            normalize_embeddings=True
        )
        
        # Benzerlikleri hesapla (normalize edilmiş vektörlerin nokta çarpımı = kosinüs)
        similarities = np.dot(self.embeddings, query_embedding)
        
        # En iyi k indeksi al
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        # Sonuçları oluştur, minimum skora göre filtrele
        results = []
        for idx in top_indices:
            score = float(similarities[idx])
            if score >= min_score:
                results.append({
                    "text": self.documents[idx],
                    "score": score,
                    "source": self.metadata[idx]["source"],
                    "index": int(idx)
                })
        
        return results

print("SimpleRAG sınıfı tanımlandı!")

In [ ]:
# Test için örnek belgeler
vacation_policy = """
Çalışanlar yılda 15 gün ücretli izin alırlar.
Kullanılmayan izin günleri, maksimum 5 güne kadar bir sonraki yıla devredilebilir.
İzin talepleri, İK portalı üzerinden en az 2 hafta önceden sunulmalıdır.
Yeni çalışanlar, 90 günlük deneme süresini tamamladıktan sonra izin kullanmaya hak kazanırlar.
"""

expense_policy = """
İş giderleri, harcama tarihinden itibaren 30 gün içinde sunulmalıdır.
50 TL üzerindeki tüm harcamalar için fiş gereklidir. İş seyahati sırasındaki yemekler günde 75 TL'ye kadar karşılanır.
Harcama raporlarını uygun belgelerle birlikte finans portalı üzerinden gönderin.
500 TL üzerindeki harcamalar için yönetici onayı gereklidir.
"""

remote_work_policy = """
Çalışanlar, yönetici onayı ile haftada 3 güne kadar uzaktan çalışabilirler.
Uzaktan çalışanlar, çekirdek saatlerde (yerel saat dilimleri ile 10:00-15:00 arası) müsait olmalıdır.
Ev ofisi ekipmanları, yönetici onayı ile 500 TL'ye kadar karşılanabilir.
Uzaktan çalışma düzenlemeleri yazılı olarak belgelenmelidir.
"""

print("Örnek belgeler oluşturuldu!")

In [ ]:
# RAG'i başlat ve belgeleri ekle
rag = SimpleRAG()

rag.add_document(vacation_policy, "izin_politikasi.txt")
rag.add_document(expense_policy, "harcama_politikasi.txt")
rag.add_document(remote_work_policy, "uzaktan_calisma_politikasi.txt")

print(f"\nBilgi tabanındaki toplam belge: {len(rag.documents)}")

In [ ]:
# Erişimi test et
query = "Kaç gün izin hakkım var?"
results = rag.retrieve(query)

print(f"Sorgu: '{query}'\n")
print("Erişilen belgeler:")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Skor: {doc['score']:.3f} | Kaynak: {doc['source']}")
    print(f"    {doc['text'][:100]}...")

In [ ]:
# Farklı bir sorguyla test et
query2 = "Yemek harcama limiti nedir?"
results2 = rag.retrieve(query2)

print(f"Sorgu: '{query2}'\n")
print("Erişilen belgeler:")
for i, doc in enumerate(results2):
    print(f"\n[{i+1}] Skor: {doc['score']:.3f} | Kaynak: {doc['source']}")
    print(f"    {doc['text'][:100]}...")

## 3. Erişilen Bağlamla İstem Oluşturma

Şimdi erişilen belgeleri kullanan istemler oluşturalım.

In [ ]:
def build_rag_prompt(query, retrieved_docs, min_score=0.3):
    """Erişilen bağlam ve alıntı talimatlarıyla bir istem oluştur."""
    # Skora göre filtrele
    good_docs = [d for d in retrieved_docs if d["score"] >= min_score]
    
    # İlgili belge bulunamadığında durumu ele al
    if not good_docs:
        return f"""Sorunuzu yanıtlamak için ilgili bilgi bulamadım.

Soru: {query}

Lütfen sorunuzu yeniden ifade edin veya genel bilgime dayanarak yanıt vermemi isterseniz bana bildirin."""
    
    # Alıntı işaretleyicileriyle bağlam oluştur
    context_parts = []
    for i, doc in enumerate(good_docs):
        source = doc.get("source", "bilinmeyen")
        context_parts.append(f"[{i+1}] (Kaynak: {source})\n{doc['text']}")
    
    context = "\n\n".join(context_parts)
    
    return f"""Soruyu yanıtlamak için aşağıdaki kaynakları kullanın.
Kaynakları [1], [2], vb. kullanarak alıntılayın. Sadece sağlanan kaynaklardan bilgi kullanın.
Kaynaklar yanıtı içermiyorsa, bunu belirtin.

Kaynaklar:
{context}

Soru: {query}

Yanıt:"""

print("build_rag_prompt() tanımlandı!")

In [ ]:
# RAG isteminin nasıl göründüğüne bak
query = "Kaç gün izin hakkım var?"
docs = rag.retrieve(query)
prompt = build_rag_prompt(query, docs)

print("RAG İSTEMİ:")
print("="*50)
print(prompt)

In [ ]:
def rag_answer(query, rag_system):
    """RAG kullanarak bir soruyu yanıtla."""
    # İlgili belgeleri eriş
    docs = rag_system.retrieve(query, top_k=3)
    
    # İstemi oluştur
    prompt = build_rag_prompt(query, docs)
    
    # Yardımcımızı kullanarak yanıt üret
    response = chat(
        prompt,
        system="Sağlanan kaynaklara dayalı olarak soruları yanıtlayan yardımcı bir asistansın. Kaynaklarını her zaman belirt.",
        temperature=0.3  # Gerçek doğruluk için düşük sıcaklık
    )
    
    return response

print("rag_answer() tanımlandı!")

In [ ]:
# RAG yanıtlamayı test et
questions = [
    "Çalışanlar kaç gün izin alır?",
    "İş seyahati için yemek harcama limiti nedir?",
    "Evden çalışabilir miyim?",
]

for q in questions:
    print(f"S: {q}")
    answer = rag_answer(q, rag)
    print(f"C: {answer}\n")
    print("-"*50 + "\n")

## 4. Araç Çağırma Temelleri

Bazen bir modelin bilgi erişmenin ötesinde daha fazlasını yapması gerekir. Eylemler alması gerekir.

In [ ]:
# Kullanılabilir araçları tanımla
# UYARI: eval() burada basitlik için kullanılmıştır. Üretimde, kod enjeksiyon
# saldırılarını önlemek için `simpleeval` gibi uygun bir matematik ayrıştırıcı kütüphanesi kullanın.
TOOLS = {
    "calculate": {
        "description": "Temel aritmetik işlemleri gerçekleştir. Girdi '2 + 2' veya '15 * 3' gibi bir matematik ifadesi olmalıdır.",
        "function": lambda expr: str(eval(expr, {"__builtins__": {}}, {}))
    },
    "get_date": {
        "description": "Güncel tarihi al.",
        "function": lambda: datetime.now().strftime("%Y-%m-%d")
    }
}

def parse_tool_call(response):
    """Model çıktısından araç çağrısını çıkar."""
    match = re.search(r'<tool>(\w+)\((.*)\)</tool>', response, re.DOTALL)
    if match:
        return match.group(1), match.group(2).strip()
    return None, None

def execute_tool(tool_name, argument):
    """Beyaz listedeki bir aracı güvenli şekilde çalıştır."""
    if tool_name not in TOOLS:
        return f"Hata: Bilinmeyen araç '{tool_name}'"
    
    try:
        if argument:
            result = TOOLS[tool_name]["function"](argument)
        else:
            result = TOOLS[tool_name]["function"]()
        return str(result)
    except Exception as e:
        return f"{tool_name} çalıştırılırken hata: {e}"

print("Araç fonksiyonları tanımlandı!")

In [ ]:
def chat_with_tools(user_message):
    """Araç çağırma yeteneği ile sohbet."""
    
    # Araç açıklamalarını oluştur
    tool_descriptions = "\n".join(
        f"- {name}: {info['description']}"
        for name, info in TOOLS.items()
    )
    
    # İlk tur: modele sor
    prompt = f"""Şu araçlara erişimin var:
{tool_descriptions}

Bir araç kullanmak için şunu yaz: <tool>isim(argüman)</tool>
Bir aracı sadece soruyu yanıtlamak için gerekiyorsa kullan.

Kullanıcı: {user_message}
Asistan:"""
    
    first_response = chat(prompt, temperature=0)  # Güvenilir ayrıştırma için deterministik
    print(f"Modelin ilk yanıtı: {first_response}")
    
    # Model bir araç kullanmak istiyor mu kontrol et
    tool_name, argument = parse_tool_call(first_response)
    
    if tool_name:
        # Aracı çalıştır
        tool_result = execute_tool(tool_name, argument)
        print(f"Araç çalıştırıldı: {tool_name}({argument}) = {tool_result}")
        
        # İkinci tur: sonucu modele geri ver
        followup = f"""{prompt}{first_response}

Araç sonucu: {tool_result}

Şimdi kullanıcıya nihai yanıtını ver:"""
        
        final_response = chat(followup, temperature=0.3)
        return final_response
    
    # Araç gerekmedi, ilk yanıtı döndür
    return first_response

print("chat_with_tools() tanımlandı!")

In [ ]:
# Araç çağırmayı test et
print("Araç çağırma test ediliyor...\n")

# Hesaplama testi
print("S: 847'nin %15'i nedir?")
answer = chat_with_tools("847'nin %15'i nedir?")
print(f"Nihai yanıt: {answer}\n")
print("-"*50)

# Tarih testi
print("\nS: Bugünün tarihi nedir?")
answer = chat_with_tools("Bugünün tarihi nedir?")
print(f"Nihai yanıt: {answer}")

## 5. Değerlendirme ve Test

RAG sisteminizin iyi çalışıp çalışmadığını nasıl anlarsınız? Bir değerlendirme seti oluşturun.

In [ ]:
def evaluate_response(response, expected_traits):
    """Bir yanıtı beklenen özelliklerle değerlendir."""
    results = {}
    response_lower = response.lower()
    
    # Alıntı işaretleyicilerini kontrol et
    if "cites_source" in expected_traits:
        has_citation = bool(re.search(r'\[\d+\]', response))
        results["cites_source"] = (has_citation == expected_traits["cites_source"])
    
    # Gerekli anahtar kelimeleri kontrol et
    if "contains_keywords" in expected_traits:
        keywords = expected_traits["contains_keywords"]
        all_present = all(kw.lower() in response_lower for kw in keywords)
        results["contains_keywords"] = all_present
    
    # Bir ret olmadığını kontrol et
    if "not_refusal" in expected_traits:
        refusal_phrases = ["bilmiyorum", "yapamam", "kaynaklarda yok", "ilgili değil"]
        is_refusal = any(phrase in response_lower for phrase in refusal_phrases)
        if expected_traits["not_refusal"]:
            results["not_refusal"] = not is_refusal
        else:
            results["not_refusal"] = is_refusal
    
    # Minimum uzunluğu kontrol et
    if "min_words" in expected_traits:
        word_count = len(response.split())
        results["min_words"] = (word_count >= expected_traits["min_words"])
    
    return results

print("evaluate_response() tanımlandı!")

In [ ]:
# Değerlendirme setini tanımla
eval_set = [
    {
        "query": "Çalışanlar kaç gün izin alır?",
        "expected_traits": {
            "contains_keywords": ["15", "gün"],
            "cites_source": True,
            "not_refusal": True
        }
    },
    {
        "query": "Yemek için harcama limiti nedir?",
        "expected_traits": {
            "contains_keywords": ["75"],
            "cites_source": True,
            "not_refusal": True
        }
    },
    {
        "query": "Hayatın anlamı nedir?",  # Kapsam dışı!
        "expected_traits": {
            "not_refusal": False,  # Reddetmeli
            "cites_source": False
        }
    },
]

print(f"Değerlendirme seti: {len(eval_set)} test senaryosu")

In [ ]:
def run_evaluation(rag_system, eval_set):
    """Tam değerlendirmeyi çalıştır ve sonuçları raporla."""
    total_tests = 0
    passed_tests = 0
    
    for case in eval_set:
        print(f"\nSorgu: {case['query']}")
        
        response = rag_answer(case["query"], rag_system)
        results = evaluate_response(response, case["expected_traits"])
        
        for trait, passed in results.items():
            total_tests += 1
            if passed:
                passed_tests += 1
                print(f"  [BAŞARILI] {trait}")
            else:
                print(f"  [BAŞARISIZ] {trait}")
        
        print(f"  Yanıt: {response[:100]}...")
    
    print(f"\n{'='*50}")
    print(f"Sonuçlar: {passed_tests}/{total_tests} test başarılı (%{100*passed_tests/total_tests:.1f})")

print("run_evaluation() tanımlandı!")

In [ ]:
# Değerlendirmeyi çalıştır
print("RAG değerlendirmesi çalıştırılıyor...")
run_evaluation(rag, eval_set)

## 6. Akış Yanıtları

Akış, üretildikçe çıktıyı göstererek yanıtların daha hızlı hissedilmesini sağlar.

In [ ]:
def stream_response(query, rag_system):
    """Ollama kullanarak RAG yanıtını akışla ilet."""
    import requests
    
    docs = rag_system.retrieve(query, top_k=3)
    prompt = build_rag_prompt(query, docs)
    
    # Ollama'nın akış uç noktasını doğrudan kullan
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "llama3.2",
            "messages": [
                {"role": "system", "content": "Sen yardımcı bir asistansın."},
                {"role": "user", "content": prompt}
            ],
            "stream": True
        },
        stream=True
    )
    
    full_response = ""
    for line in response.iter_lines():
        if line:
            data = json.loads(line)
            if "message" in data and "content" in data["message"]:
                content = data["message"]["content"]
                print(content, end="", flush=True)
                full_response += content
    
    print()  # Sonda yeni satır
    return full_response

print("stream_response() tanımlandı!")

In [ ]:
# Akışı test et
print("Akış yanıtı test ediliyor...\n")
print("S: Kaç gün uzaktan çalışabilirim?\n")
print("C: ", end="")
stream_response("Kaç gün uzaktan çalışabilirim?", rag)

## Alıştırmalar

### Alıştırma 1: Kendi Bilgi Tabanınızı Oluşturun

İlgilendiğiniz bir konu hakkında bir bilgi tabanı oluşturun.

In [ ]:
# KODUNUZ BURAYA
# 1. Bir konu hakkında 10 kısa belge yazın (tarifler, oyun kuralları, ders notları)
# 2. Bir SimpleRAG örneği başlatın
# 3. Belgelerinizi ekleyin
# 4. 5 soruyla test edin
# 5. Gözlemleyin: Doğru parçaları erişiyor mu?

### Alıştırma 2: Sıcaklık Deneyi

Sıcaklığın RAG yanıtlarını nasıl etkilediğini test edin.

In [ ]:
# KODUNUZ BURAYA
# 1. rag_answer() fonksiyonunu bir sıcaklık parametresi kabul edecek şekilde değiştirin
# 2. Aynı soruyu 0.3, 0.7, 1.0 sıcaklıklarıyla sorun
# 3. Yanıtları karşılaştırın
# 4. Gerçek sorular için hangisi en güvenilir?

### Alıştırma 3: Değerlendirme Seti Oluşturma

Bilgi tabanınız için kapsamlı bir değerlendirme seti oluşturun.

In [ ]:
# KODUNUZ BURAYA
# 1. Bilgi tabanınız için 10 değerlendirme sorusu oluşturun
# 2. Şunları ekleyin: 5 kolay, 3 zor, 2 düşmanca (kapsam dışı)
# 3. Her biri için beklenen özellikleri tanımlayın
# 4. Değerlendirmeyi çalıştırın ve doğruluğu raporlayın

### Alıştırma 4: Kontrol Noktası - Kişisel Bilgi Asistanı

Tam bir "belgelerinizle sohbet" uygulaması geliştirin.

In [ ]:
# KODUNUZ BURAYA
# 1. 5-10 metin dosyası toplayın (notlar, makaleler, belgeler)
# 2. Bunları bir SimpleRAG örneğine yükleyin
# 3. Her yanıt için RAG kullanan bir sohbet döngüsü oluşturun
# 4. Şunları dahil edin: token sayma, geçmiş yönetimi, alıntılar
# 5. 10 soruluk bir değerlendirme seti oluşturun
# 6. Değerlendirmeyi çalıştırın ve doğruluğu raporlayın

## Özet

**Geliştirdiklerimiz:**

- Geçmiş yönetimi ve token bütçelemesi olan bir sohbet döngüsü
- Belge parçalamadan alıntı üretimine kadar eksiksiz bir RAG sistemi
- Güvenlik konularıyla birlikte araç çağırma önizlemesi
- Üretim hazırlığı için değerlendirme ve doğrulama

**Öğrendiklerimiz:**

- Eğitim bilginiz (gömmeler, parçalama, veri kalitesi, tekrarlanabilirlik) doğrudan uygulamalara aktarılır
- RAG, "model verimi bilmiyor" problemini çözer
- Araç çağırma, modellerin yapabileceklerini genişletir, ancak dikkatli güvenlik gerektirir
- Değerlendirme isteğe bağlı değil, zorunludur